# PP-OCRv5 Recognition Fine-Tuning Notebook

This notebook mirrors `app/scripts/finetune_ppocrv5_rec.py`:
1. Configure dataset/model/output paths
2. Validate PaddleOCR-native dataset layout
3. Build training command
4. Run training
5. Inspect saved checkpoints


In [ ]:
from __future__ import annotations

import argparse
import shlex
import subprocess
import sys
from pathlib import Path

# Resolve project root so `app.*` imports work from different notebook launch dirs.
candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next((p for p in candidates if (p / 'app').exists()), Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from app.scripts.finetune_ppocrv5_rec import (
    build_training_command,
    require_exists,
    resolve_training_device,
    validate_dataset_layout,
)

print(f'Project root: {PROJECT_ROOT}')


## 1) Configure paths and hyperparameters

Dataset format must be:
- `train_images/`, `val_images/`
- `train_label.txt`, `val_label.txt`
- each label line: `relative_path<TAB>text`


In [ ]:
# Edit these values
DATA_ROOT = PROJECT_ROOT / 'data' / 'ppocrv5_dataset'
OUTPUT_DIR = PROJECT_ROOT / 'artifacts' / 'ppocrv5_finetune_run1'
PRETRAINED_MODEL = PROJECT_ROOT / 'models' / 'ppocrv5' / 'main'
BASE_CONFIG = PROJECT_ROOT / 'configs' / 'rec' / 'PP-OCRv5' / 'rec_ppocr_v5_train.yml'

EPOCHS = 30
BATCH_SIZE = 32
LEARNING_RATE = 5e-4
DEVICE = 'auto'  # one of: auto, cpu, mps, cuda

RUN_TRAINING = False  # set True when validation passes

print('DATA_ROOT:', DATA_ROOT)
print('OUTPUT_DIR:', OUTPUT_DIR)
print('PRETRAINED_MODEL:', PRETRAINED_MODEL)
print('BASE_CONFIG:', BASE_CONFIG)


## 2) Validate dataset and required files

In [ ]:
layout, stats = validate_dataset_layout(DATA_ROOT)
require_exists(PRETRAINED_MODEL, 'pretrained model')
require_exists(BASE_CONFIG, 'base config')

print('Validation passed')
print('train_samples:', stats['train_samples'])
print('val_samples:', stats['val_samples'])


In [ ]:
# Quick label preview
print('train_label.txt sample:')
for line in layout.train_label_file.read_text(encoding='utf-8').splitlines()[:5]:
    print('  ', line)

print('\nval_label.txt sample:')
for line in layout.val_label_file.read_text(encoding='utf-8').splitlines()[:5]:
    print('  ', line)


## 3) Build PaddleOCR training command

In [ ]:
training_device = resolve_training_device(DEVICE)
args = argparse.Namespace(
    data_root=DATA_ROOT,
    output_dir=OUTPUT_DIR,
    pretrained_model=PRETRAINED_MODEL,
    base_config=BASE_CONFIG,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    device=DEVICE,
)
command = build_training_command(args, layout, training_device)

print('device_requested:', DEVICE)
print('device_used:', training_device)
print('command:')
print('  ' + shlex.join(command))


## 4) Launch training (optional)

Set `RUN_TRAINING = True` above to execute training.

In [ ]:
if RUN_TRAINING:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    subprocess.run(command, check=True, cwd=PROJECT_ROOT)
    print('Training completed.')
else:
    print('RUN_TRAINING is False. Skipping training launch.')


## 5) Inspect saved artifacts

PaddleOCR writes checkpoints and best model under `OUTPUT_DIR`.

In [ ]:
if OUTPUT_DIR.exists():
    files = sorted(OUTPUT_DIR.rglob('*'))
    print(f'Found {len(files)} files/directories under {OUTPUT_DIR}')
    for p in files[:80]:
        rel = p.relative_to(OUTPUT_DIR)
        print(rel)
    if len(files) > 80:
        print('... truncated ...')
else:
    print(f'Output directory does not exist yet: {OUTPUT_DIR}')
